# 02 — Data Preprocessing

## 1. Objective

The goal of this notebook is to prepare clean and consistent versions of the training and test datasets for later modeling.

This notebook will:

- load the raw training and test data;
- identify the target, identifier, categorical, and numerical columns;
- remove or handle non-predictive identifier information;
- prepare the target variable for modeling;
- check consistency between training and test data;
- save cleaned datasets for reuse in later modeling notebooks.

Model-specific transformations such as one-hot encoding and scaling will be handled later inside the modeling pipeline to avoid data leakage.

## 2. Load Raw Data

In [1]:
import pandas as pd

train = pd.read_csv("../data/Train.csv")
test = pd.read_csv("../data/Test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())

Train shape: (23524, 13)
Test shape: (10086, 12)


,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed


,country,year,uniqueid,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_6056,Urban,Yes,3,30,Male,Head of Household,Married/Living together,Secondary education,Formally employed Government
1,Kenya,2018,uniqueid_6060,Urban,Yes,7,51,Male,Head of Household,Married/Living together,Vocational/Specialised training,Formally employed Private
2,Kenya,2018,uniqueid_6065,Rural,No,3,77,Female,Parent,Married/Living together,No formal education,Remittance Dependent
3,Kenya,2018,uniqueid_6072,Rural,No,6,39,Female,Head of Household,Married/Living together,Primary education,Remittance Dependent
4,Kenya,2018,uniqueid_6073,Urban,No,3,16,Male,Child,Single/Never Married,Secondary education,Remittance Dependent


## 3. Define Target, Identifier, Numerical, and Categorical Columns

The dataset contains one target column, one identifier column, numerical features, and categorical features.

We define them explicitly so that later preprocessing and modeling steps are clear and reproducible.

In [2]:
target_column = "bank_account"
id_column = "uniqueid"

numerical_features = [
    "year",
    "household_size",
    "age_of_respondent"
]

categorical_features = [
    "country",
    "location_type",
    "cellphone_access",
    "gender_of_respondent",
    "relationship_with_head",
    "marital_status",
    "education_level",
    "job_type"
]

print("Target:", target_column)
print("Identifier:", id_column)
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


Target: bank_account
Identifier: uniqueid
Numerical features: ['year', 'household_size', 'age_of_respondent']
Categorical features: ['country', 'location_type', 'cellphone_access', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']


## 4. Check Train/Test Structure

The training data contains the target column `bank_account`, while the test data contains only the input features.

We verify that both datasets contain the same predictive features.

In [21]:
train_features = set(train.columns) - {target_column}
test_features = set(test.columns)

print("Target only in train:", target_column in train.columns and target_column not in test.columns)
print("Same feature columns:", train_features == test_features)

Target only in train: True
Same feature columns: True


## 5. Handle Identifier Column

`uniqueid` is an identifier rather than a meaningful predictive feature.

We keep it separately for traceability and submission purposes, but exclude it from the model features.

In [13]:
train_ids = train["uniqueid"].copy()
test_ids = test["uniqueid"].copy()

train_clean = train.drop(columns=["uniqueid"]).copy()
test_clean = test.drop(columns=["uniqueid"]).copy()

print("Train clean shape:", train_clean.shape)
print("Test clean shape:", test_clean.shape)

Train clean shape: (23524, 12)
Test clean shape: (10086, 11)


## 6. Prepare Target Variable

The target variable `bank_account` contains two categories: `Yes` and `No`.

For modeling, we convert the target to binary values:

- `No` → 0
- `Yes` → 1

In [14]:
train_clean["bank_account"] = train_clean["bank_account"].map({
    "No": 0,
    "Yes": 1
})

print(train_clean["bank_account"].value_counts())

bank_account
0    20212
1     3312
Name: count, dtype: int64


## 7. Final Cleaning Checks

Before saving the cleaned datasets, we verify that:

- `uniqueid` has been removed from the model-ready data;
- the target has been converted to binary values;
- no missing values were introduced during preprocessing;
- training and test features remain consistent.

In [ ]:
print("Train clean shape:", train_clean.shape)
print("Test clean shape:", test_clean.shape)

print("\nMissing values in train:")
print(train_clean.isna().sum().sum())

print("\nMissing values in test:")
print(test_clean.isna().sum().sum())

print("\nTarget values:")
print(train_clean["bank_account"].unique())


Train clean shape: (23524, 12)
Test clean shape: (10086, 11)

Missing values in train:
0

Missing values in test:
0

Target values:
[1 0]

Unique ID in train_clean: False
Unique ID in test_clean: False


## 8. Save Clean Data

The cleaned training and test datasets are saved so they can be reused directly in later modeling notebooks.

Model-specific transformations such as encoding and scaling will still be handled inside the modeling pipeline.

In [19]:
train_clean.to_csv("../data/Train_clean.csv", index=False)
test_clean.to_csv("../data/Test_clean.csv", index=False)

print("Clean datasets saved successfully.")

Clean datasets saved successfully.


In [20]:
saved_train = pd.read_csv("../data/Train_clean.csv")
saved_test = pd.read_csv("../data/Test_clean.csv")

print("Saved train shape:", saved_train.shape)
print("Saved test shape:", saved_test.shape)

Saved train shape: (23524, 12)
Saved test shape: (10086, 11)


## 9. Preprocessing Strategy

The saved datasets are cleaned but not yet fully transformed for modeling.

Reusable model preprocessing will be defined separately in `src/preprocessing.py`, including:

- one-hot encoding for categorical features;
- scaling for numerical features when required;
- a consistent preprocessing pipeline that can be reused across baseline and advanced models.

The preprocessing pipeline will be fitted only after the train/validation split to avoid data leakage.